In [5]:
import argparse
import os

import pandas as pd
import pickle
from tqdm import tqdm
from copy import deepcopy

import numpy as np
import torch
from torch import optim
from torch.utils.tensorboard import SummaryWriter

import matplotlib.pyplot as plt

import sys

In [6]:
d1 = 5000
r = 5
d2 = 100

mat1 = np.random.rand(d1, r)
mat2 = np.random.rand(r, d2)
M = mat1 @ mat2

In [7]:
from data_utils import load_data_all

dataset = 'ml-1m'
M = load_data_all(dataset)

ml-1m


NameError: name 'args' is not defined

In [ ]:
def differential_private_initialization(Y, G, tau, sigma0, p):
    m, n = Y.shape

    def projection(V, alpha2):
        V_projected = np.copy(V)
        for i in range(V_projected.shape[0]):
            row_norm = np.linalg.norm(V_projected[i, :], 2)
            if row_norm > alpha2:
                V_projected[i, :] = (alpha2 / row_norm) * V_projected[i, :]
        return V_projected

    # Step 1: Projection
    Y = projection(Y, G)

    # Step 2: Communicate (receive)
    Y_i = Y  # All users send their data to the server

    # Step 4: Server calculates private covariance matrix A
    A = tau**2 * np.sum([np.outer(Y_i[i], Y_i[i]) for i in range(m)], axis=0) / p**2 + np.random.normal(0, sigma0, size=(n, n))
    #A = (A + A.T) / 2  # Ensure A is symmetric

    # Step 5: Obtain (V^0, Σ) by performing the rank r SVD of A
    _, S, Vt = np.linalg.svd(A, full_matrices=False)

    V0 = Vt[:r, :].T
    Sigma = np.diag(S[:r])
    V0_tilde = V0 * np.sqrt(S[:r])

    U0 = np.array([(tau * Y[i].T @ V0 * np.sqrt(S[:r])/p) for i in range(m)])

    return U0.T, V0_tilde

# 初始化参数
p = 0.5
m, n = M.shape
mask = np.random.rand(m, n) <= p
Y = M * mask   # 示例数据
G = 1.0
tau = 0.1
sigma0 = 0.1


U0, V0_tilde = differential_private_initialization(Y, G, tau, sigma0, p)
print("Initialized U0^T:\n", U0)
U0 = U0.T

Initialized U0^T:
 [[-1.38391293 -1.23199438 -1.41648845 ... -1.51106775 -1.35849529
  -1.44208262]
 [ 0.03621034 -0.03942174 -0.02161917 ...  0.04746605  0.01215617
  -0.0029639 ]
 [ 0.01783605 -0.02517933 -0.0056624  ...  0.00531876  0.0161934
  -0.01428256]
 [-0.00921434 -0.0178352  -0.03610846 ...  0.02632771 -0.01027971
   0.00533439]
 [-0.0087846   0.0087025  -0.0059898  ... -0.04351716  0.00185672
  -0.00371313]]


In [ ]:
def P_omega(X):
    return X * mask

def P_omegai(X, i):
    return X * mask[i]

In [ ]:
import numpy as np

def differential_private_low_rank_matrix_completion(U0, V0_tilde, Y, T, alpha1, alpha2, G, eta, sigma1, sigma2):
    """
    Ut -> U
    """

    U = U0
    V = V0_tilde
    Y_hat = np.array([V @ U[i] for i in range(m)])

    def P_omega(X):
        mask = Y == 0
        return X * mask
    
    def P_omegai(X, i):
        mask = (Y == 0 )[i]
        return X * mask

    
    def projection(X, alpha):
        norm = np.linalg.norm(X, ord=2, axis=None)
        return X if norm <= alpha else (alpha / norm) * X
    
    def projection_(V, alpha):

        V_projected = np.copy(V)
        for i in range(V_projected.shape[0]):
            row_norm = np.linalg.norm(V_projected[i, :], 2)
            if row_norm > alpha2:
                V_projected[i, :] = (alpha2 / row_norm) * V_projected[i, :]
        return V_projected
    
    def projection_1d(V, alpha2):

        row_norm = np.linalg.norm(V, 2)
        if row_norm > alpha2:
            V = (alpha2 / row_norm) * V
        return V

    for t in tqdm(range(T)):
        if t == 0:            
            # 用户端接收
            P_omega_Y_hat_minus_Y = P_omega(Y_hat - Y)

        # 服务器端计算
        P_omega_Y_hat_minus_Y = projection(P_omega_Y_hat_minus_Y, G)

        R_tilde = np.sum([U[i] @ U[i].T for i in range(m)], axis=0) - V.T  @ V + np.random.normal(0, sigma1, size=(r, r))
        V = V - (eta / p) * (P_omega_Y_hat_minus_Y.T @ U + np.random.normal(0, sigma2, size=(n, r))) + (eta / 2 ) * V @ R_tilde
        V = projection(V, alpha2)

        # 用户端更新
        for i in range(m):
            U[i] = U[i] - (eta / p) * P_omegai(Y_hat[i].T - Y[i].T, i) @ V - (eta / 2) * U[i].T @ R_tilde
            U[i] = projection_1d(U[i], alpha1)
            Y_hat[i] = V @ U[i].T


    return U, V

# 初始化参数
U0, V0_tilde = differential_private_initialization(Y, G, tau, sigma0, p)
U0 = U0.T
T = 30
alpha1 = 3
alpha2 = 3
G = alpha1 * alpha2
eta = 0.3
sigma1 = 0.1
sigma2 = 0.1

print(U0.shape)
print(V0_tilde.shape)

#print("Initialized U0^T:\n", U0)

U_final, V_final = differential_private_low_rank_matrix_completion(U0, V0_tilde, Y, T, alpha1, alpha2, G, eta, sigma1, sigma2)
print("Final U:\n", U_final)
print("Final V:\n", V_final)


(5000, 5)
(100, 5)


100%|██████████| 30/30 [00:32<00:00,  1.10s/it]

Final U:
 [[-1.34164137 -1.34164179 -1.34164211 -1.34163913 -1.34163954]
 [-1.34164137 -1.34164179 -1.34164211 -1.34163912 -1.34163954]
 [-1.34164137 -1.34164179 -1.34164211 -1.34163912 -1.34163954]
 ...
 [-1.34164137 -1.34164179 -1.34164211 -1.34163912 -1.34163954]
 [-1.34164137 -1.34164179 -1.34164211 -1.34163913 -1.34163954]
 [-1.34164137 -1.34164179 -1.34164211 -1.34163912 -1.34163954]]
Final V:
 [[-0.12724149 -0.12724195 -0.12724012 -0.12723935 -0.12724481]
 [-0.10968641 -0.10968475 -0.1096824  -0.10968482 -0.10968177]
 [-0.10210297 -0.10210024 -0.10210175 -0.10210485 -0.10210187]
 [-0.05251024 -0.05251206 -0.05251097 -0.05250915 -0.05250745]
 [-0.11944624 -0.11945164 -0.11944568 -0.11944463 -0.1194453 ]
 [-0.09390301 -0.09390362 -0.09390459 -0.09390259 -0.09390359]
 [-0.13092415 -0.13092413 -0.13092789 -0.13092553 -0.13092672]
 [-0.10431343 -0.10430819 -0.10431065 -0.10430586 -0.10430741]
 [-0.07644151 -0.07644451 -0.07644083 -0.07644331 -0.07644433]
 [-0.11120316 -0.11119692 -0.

In [ ]:
X_estimation = U_final @ V_final.T
(np.linalg.norm(P_omega(M - X_estimation))**2)/(m*n)

0.2042864388637064

In [ ]:
MTM = M.T @ M
VTV = V_final @ V_final.T
print(np.linalg.norm(VTV-MTM) / np.linalg.norm(MTM))

0.9999910114180205


In [ ]:
XTX = X_estimation.T @ X_estimation
print(np.linalg.norm(XTX-MTM) / np.linalg.norm(MTM))

0.6187920708441681
